In [3]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
ds=pd.read_csv("clean_salary_data.csv")

In [83]:
ds.drop(columns=['Unnamed: 0'], inplace=True)
ds

,Job_Role,Company,Year_Joined,Experience,Qualification,Location,Salary
0,Software Engineer,Amazon,2021,5.0,12th Pass,Bangalore,500000.0
1,Data Scientist,Google,2009,7.0,Masters,Delhi,580000.0
2,Data Analyst,TCS,2001,14.0,PhD,Delhi,860000.0
3,Data Scientist,Accenture,1996,27.0,Graduate,Delhi,1380000.0
4,Web Developer,Google,2005,26.0,12th Pass,Hyderabad,1340000.0
5,HR,Infosys,2002,19.0,Graduate,Hyderabad,1060000.0
6,Software Engineer,Wipro,2014,13.0,Graduate,Pune,920000.0
7,Developer,Google,2012,8.0,Graduate,Bangalore,620000.0
8,HR,Google,2020,20.0,Bachelors,Mumbai,1200000.0
9,HR,Microsoft,2018,25.0,PhD,Chennai,1400000.0


In [84]:
ds.columns

Index(['Job_Role', 'Company', 'Year_Joined', 'Experience', 'Qualification',
       'Location', 'Salary'],
      dtype='object')

In [85]:
#Lets divide in independent(X) and dependent(y)
X=ds[['Job_Role', 'Company', 'Year_Joined', 'Experience', 'Qualification', 'Location']]
y= ds[['Salary']]

In [86]:
X

,Job_Role,Company,Year_Joined,Experience,Qualification,Location
0,Software Engineer,Amazon,2021,5.0,12th Pass,Bangalore
1,Data Scientist,Google,2009,7.0,Masters,Delhi
2,Data Analyst,TCS,2001,14.0,PhD,Delhi
3,Data Scientist,Accenture,1996,27.0,Graduate,Delhi
4,Web Developer,Google,2005,26.0,12th Pass,Hyderabad
5,HR,Infosys,2002,19.0,Graduate,Hyderabad
6,Software Engineer,Wipro,2014,13.0,Graduate,Pune
7,Developer,Google,2012,8.0,Graduate,Bangalore
8,HR,Google,2020,20.0,Bachelors,Mumbai
9,HR,Microsoft,2018,25.0,PhD,Chennai


In [87]:
y

,Salary
0,500000.0
1,580000.0
2,860000.0
3,1380000.0
4,1340000.0
5,1060000.0
6,920000.0
7,620000.0
8,1200000.0
9,1400000.0


In [88]:
len(X["Company"].unique())

9

In [89]:
len(X["Job_Role"].unique())

9

In [90]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder()
ohe.fit(X[['Company','Job_Role','Year_Joined']])

OneHotEncoder()

In [91]:
#make column transformer
from sklearn.compose import make_column_transformer
ct=make_column_transformer((OneHotEncoder(categories=ohe.categories_),['Company','Job_Role','Year_Joined']),remainder='passthrough',force_int_remainder_cols=False)

In [92]:
#Lets make model object
from sklearn.linear_model import LinearRegression
reg=LinearRegression()

In [93]:
# Lets make pipeline whose first part is CT and second is model
from sklearn.pipeline import make_pipeline
pipe=make_pipeline(ct,reg)

In [106]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Step 1: Load dataset
ds = pd.read_csv("clean_salary_data.csv")  # Make sure this file exists in your working directory

# Step 2: Prepare features (X) and target (y)
X = ds.drop("Salary", axis=1)
y = ds["Salary"]

# Step 3: Identify categorical columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Step 4: Create preprocessor for encoding categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'  # Keep numerical columns like Experience, Year_Joined
)

# Step 5: Build pipeline using RandomForestRegressor
pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Step 6: Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Step 7: Train the model
pipe.fit(X_train, y_train)

# Step 8: Make predictions
y_pred = pipe.predict(X_test)

# Step 9: Calculate and print R² score
score = r2_score(y_test, y_pred)
print("✅ R² Score on your dataset:", round(score, 4))

# Step 10: Display actual vs predicted salaries
results = pd.DataFrame({
    "Job_Role": X_test["Job_Role"].values,
    "Company": X_test["Company"].values,
    "Actual Salary": y_test.values,
    "Predicted Salary": y_pred.round(2)
})

print("\n🔍 Sample Predictions:")
print(results.head(10))


✅ R² Score on your dataset: 0.2287

🔍 Sample Predictions:
            Job_Role  Company  Actual Salary  Predicted Salary
0     Database Admin  Infosys      1250000.0         1255200.0
1        ML Engineer   Amazon      1350000.0          746900.0
2  Software Engineer    Wipro       450000.0          722400.0
3     Database Admin    Wipro       680000.0          734500.0


In [107]:
index=np.argmax(score)
index

0

In [108]:
ds.drop(columns=['Unnamed: 0'], inplace=True)
ds.columns


Index(['Job_Role', 'Company', 'Year_Joined', 'Experience', 'Qualification',
       'Location', 'Salary'],
      dtype='object')

In [109]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=index)
pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Job_Role', 'Company',
                                                   'Qualification',
                                                   'Location'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [111]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Prepare features and target
X = ds.drop("Salary", axis=1)
y = ds["Salary"]

# Identify categorical columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Preprocessor for categorical columns
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols)],
    remainder='passthrough'
)

# Create pipeline with Random Forest
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Fit the model
pipe.fit(X_train, y_train)

# Optional: Evaluate
y_pred = pipe.predict(X_test)
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.3659875628635151


In [113]:
# Now test new input
data = [["Software Engineer", "Amazon", 2009, 5.0, "12th Pass", "Bangalore"]]
columns = ['Job_Role', 'Company', 'Year_Joined', 'Experience', 'Qualification', 'Location']
myinput = pd.DataFrame(data=data, columns=columns)
# Predict
result = pipe.predict(myinput)
print("Predicted Salary:", round(result[0]))


Predicted Salary: 613000


In [114]:
#lets exportpipe for  future use in API / web applications
import pickle as pkl
pkl.dump(pipe,open("Salary_CPP.pkl","wb+"))